# Exception Tables

In [1]:
import glob, os, json, subprocess, re, shutil
from scripts.constants import Template, ModuleNamePrefix
from scripts.lazy import getTemplateFilenamePath
from run import makeWorkingFolder
import pandas as pd, numpy as np

In [2]:
with open('reports/exceptionTrialBench/exceptionTrialBench.json', 'r') as file:
    exceptionTrialJSON = json.load(file)
exceptionTrials = pd.DataFrame(exceptionTrialJSON['exceptionTrials']).rename(columns={'None': 'Error'})
exceptionFixability = pd.DataFrame(exceptionTrialJSON['exceptionFixability']).rename(columns={'None': 'Error'})
exceptionTrialCount = pd.DataFrame(exceptionTrialJSON['exceptionTrialCount']).rename(columns={'None': 'Error'})

# exceptionTrialAll = exceptionTrials.copy()

In [3]:
exceptionTrialAll = exceptionTrials.agg(['sum']).rename(index={'sum': 'Total Occurance'})
exceptionTrialAllStyle = exceptionTrialAll.style
exceptionTrialAllStyle

,UNOPTFLAT,UNUSEDSIGNAL,TB Failures,GENUNNAMED,UNDRIVEN,WIDTHEXPAND,WIDTHTRUNC,MULTIDRIVEN,PROCASSWIRE,Error
Total Occurance,22,32,21,1,5,60,8,3,7,3


In [4]:
# exceptionTrialAll.apply(lambda x: x > 0).agg(['sum']).rename(index={'sum': 'Modules with Exception'})

In [5]:
# exceptionFixabilityProp = exceptionFixability.map(lambda x: str(x)) + '/' + exceptionTrialCount.map(lambda x: str(x))
exceptionFixabilityProp = exceptionFixability/exceptionTrialCount
exceptionFixabilityPropSum = exceptionFixabilityProp.transform(lambda x: np.isfinite(x)).agg(['sum']).map(lambda x: str(x))
exceptionFixabilityPropFinal = exceptionFixabilityProp.transform(lambda x: x > 0).agg(['sum']).map(lambda x: str(x)) + '/' + exceptionFixabilityPropSum
# exceptionTrialAll.join(exceptionFixabilityPropFinal.rename(index={'sum': 'Fixability'}))
exceptionTrialAllStyle.concat(exceptionFixabilityPropFinal.rename(index={'sum': 'Fixability'}).style)

,UNOPTFLAT,UNUSEDSIGNAL,TB Failures,GENUNNAMED,UNDRIVEN,WIDTHEXPAND,WIDTHTRUNC,MULTIDRIVEN,PROCASSWIRE,Error
Total Occurance,22,32,21,1,5,60,8,3,7,3
Fixability,1/2,3/3,5/8,1/1,0/3,6/9,2/3,0/2,2/2,1/1


In [6]:
maxModuleException = exceptionTrials.idxmax()
# print(maxModuleException)
# maxModuleException.to_dict
maxModuleExceptionDF = pd.DataFrame([maxModuleException.to_dict()],  index=['Module with Max Occurance'])
exceptionTrialAllStyle.concat(maxModuleExceptionDF.style)

,UNOPTFLAT,UNUSEDSIGNAL,TB Failures,GENUNNAMED,UNDRIVEN,WIDTHEXPAND,WIDTHTRUNC,MULTIDRIVEN,PROCASSWIRE,Error
Total Occurance,22,32,21,1,5,60,8,3,7,3
Fixability,1/2,3/3,5/8,1/1,0/3,6/9,2/3,0/2,2/2,1/1
Module with Max Occurance,adder_8bit,adder_8bit,multi_pipe_4bit,adder_8bit,adder_32bit,alu,adder_32bit,multi_pipe_4bit,asyn_fifo,asyn_fifo


In [7]:
exceptionTrialAllTranspose = exceptionTrialAllStyle.data.transpose()

for exceptionTrialAllStyleConcat in exceptionTrialAllStyle.concatenated:
    exceptionTrialAllTranspose = exceptionTrialAllTranspose.join(exceptionTrialAllStyleConcat.data.transpose())
exceptionTrialAllTranspose

,Total Occurance,Fixability,Module with Max Occurance
UNOPTFLAT,22,1/2,adder_8bit
UNUSEDSIGNAL,32,3/3,adder_8bit
TB Failures,21,5/8,multi_pipe_4bit
GENUNNAMED,1,1/1,adder_8bit
UNDRIVEN,5,0/3,adder_32bit
WIDTHEXPAND,60,6/9,alu
WIDTHTRUNC,8,2/3,adder_32bit
MULTIDRIVEN,3,0/2,multi_pipe_4bit
PROCASSWIRE,7,2/2,asyn_fifo
Error,3,1/1,asyn_fifo


## Format

In [8]:
# all_module_join_style_df.apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen'], axis=1)

## .

In [9]:
exceptionTrialAllTransposeStyle = exceptionTrialAllTranspose.style

In [10]:
a = np.array([[1,2], [3,4]])
# type(a[0])
a[0]

array([1, 2])

In [11]:
def tolOccHighlight_max(s: pd.core.series.Series):
    allFormatMaxtrix = np.array([["", 0, ""]] * len(s), dtype='object')
    allFormatMaxtrix[:, 0] = s.index
    allFormatMaxtrix[:, 1] = s.values

    maxRows = np.where(allFormatMaxtrix[:, 1] == s.max())[0]
    allFormatMaxtrix[maxRows, 2] = 'color:black;background-color:lightblue'

    minRows = np.where(allFormatMaxtrix[:, 1] == s.min())[0]
    allFormatMaxtrix[minRows, 2] = 'color:black;background-color:lightgreen'
    
    return allFormatMaxtrix[:, 2]

# def formatFixable(x:str):
#     print(x)
#     return x
def fixableHighlight_max(s: pd.core.series.Series):
    # sFormatValues = s.values
    sFormat = s.str.split('/').values
    sFormat = np.array(sFormat.tolist(), np.int32)
    sFormat = sFormat[:,0] / sFormat[:,1]
    
    allFormatMaxtrix = np.array([["", 0, ""]] * len(s), dtype='object')
    allFormatMaxtrix[:, 0] = s.index
    allFormatMaxtrix[:, 1] = sFormat

    maxRows = np.where(allFormatMaxtrix[:, 1] == np.max(sFormat))[0]
    allFormatMaxtrix[maxRows, 2] = 'color:black;background-color:lightblue'

    minRows = np.where(allFormatMaxtrix[:, 1] == np.min(sFormat))[0]
    allFormatMaxtrix[minRows, 2] = 'color:black;background-color:lightgreen'
    
    return allFormatMaxtrix[:, 2]
    # print([['']] * len(s))
    # return [''] * len(s)

In [12]:
exceptionTrialAllTransposeStyle = exceptionTrialAllTransposeStyle.apply(tolOccHighlight_max, subset=['Total Occurance'])
exceptionTrialAllTransposeStyle = exceptionTrialAllTransposeStyle.apply(fixableHighlight_max, subset=['Fixability'])

In [13]:
exceptionTrialAllTransposeStyle.format_index(escape="latex")
exceptionTrialAllTransposeStyle.format(escape="latex")

,Total Occurance,Fixability,Module with Max Occurance
UNOPTFLAT,22,1/2,adder\_8bit
UNUSEDSIGNAL,32,3/3,adder\_8bit
TB Failures,21,5/8,multi\_pipe\_4bit
GENUNNAMED,1,1/1,adder\_8bit
UNDRIVEN,5,0/3,adder\_32bit
WIDTHEXPAND,60,6/9,alu
WIDTHTRUNC,8,2/3,adder\_32bit
MULTIDRIVEN,3,0/2,multi\_pipe\_4bit
PROCASSWIRE,7,2/2,asyn\_fifo
Error,3,1/1,asyn\_fifo


In [14]:
exceptionTrialAllTransposeStyle.map_index(
    lambda v: "font-weight: bold;", axis="columns", level=0
)

,Total Occurance,Fixability,Module with Max Occurance
UNOPTFLAT,22,1/2,adder\_8bit
UNUSEDSIGNAL,32,3/3,adder\_8bit
TB Failures,21,5/8,multi\_pipe\_4bit
GENUNNAMED,1,1/1,adder\_8bit
UNDRIVEN,5,0/3,adder\_32bit
WIDTHEXPAND,60,6/9,alu
WIDTHTRUNC,8,2/3,adder\_32bit
MULTIDRIVEN,3,0/2,multi\_pipe\_4bit
PROCASSWIRE,7,2/2,asyn\_fifo
Error,3,1/1,asyn\_fifo


In [15]:
latex = exceptionTrialAllTransposeStyle.to_latex(convert_css=True, hrules=True, column_format="c|c|c|c") # "c|cc|cc|cc"
latex_list = latex.splitlines()
mid_bot_rule_index = (latex_list.index("\\midrule"), latex_list.index("\\bottomrule") - 1)
for i in range(mid_bot_rule_index[0] + 1, mid_bot_rule_index[1]):
    latex_list[i] += " \\midrule"

with open(f'reports/exceptionTrialBench/exceptionTrialBench.tex', 'w+') as file:
    print('\n'.join(latex_list), file=file)

# OpenAI Assistant Langchain

In [ ]:
from langchain.agents.openai_assistant import OpenAIAssistantRunnable
from langchain.agents import AgentExecutor
import os
from langchain_core.utils.function_calling import convert_to_openai_tool
from dotenv import load_dotenv
_ = load_dotenv()

In [ ]:
tools = [convert_to_openai_tool(tool) for tool in [{"type": "file_search"}]]
openai_assistant = OpenAIAssistantRunnable(assistant_id=os.environ['OPENAI_ASSISTANT_API_KEY'], as_agent=True)

In [ ]:
openai_assistant.invoke({"content": "Who are you?"})

# Table Report

In [ ]:
!pip install pandas

In [ ]:
import glob, os, ast, json
import pandas as pd, numpy as np

In [ ]:
llm_model="gpt-4o-mini-2024-07-18"
llm_model_alias_dict = {"gpt-4o-mini-2024-07-18": "GPT‑4o mini + COMBA-PROMPT"}
modulePaths = ['modules/*']
moduleGlobPaths = []
for modulePath in modulePaths:
    moduleGlobPaths += glob.glob(modulePath)

moduleNormPaths = [os.path.normpath(modulePath) for modulePath in moduleGlobPaths]

## RTLLM Raw Text Prompt

### 4o Mini

In [ ]:
with open('reports/assets/raw.4omini.txt', 'r') as file:
    raw_rtllm_4o_mini = json.load(file)
all_module_raw_rtllm_4omini_df = pd.DataFrame()

In [ ]:
for moduleNormPath in moduleNormPaths:
    moduleName = os.path.basename(moduleNormPath)
    if moduleName in raw_rtllm_4o_mini:
        row_data = np.array(list(raw_rtllm_4o_mini[moduleName].values())) / 5
        cur_module_df = pd.DataFrame([row_data], 
                                 index=pd.Index([moduleName], name='Designs'), 
                                 columns=pd.MultiIndex.from_product([['GPT‑4o mini + RTLLM '],['Syntax', 'Func.']]))
        all_module_raw_rtllm_4omini_df = pd.concat([all_module_raw_rtllm_4omini_df, cur_module_df])
    else:
        print(f'Module "{moduleName}" not found')
all_module_raw_rtllm_4omini_df

### o3 Mini

In [ ]:
with open('reports/assets/raw.o3mini.txt', 'r') as file:
    raw_rtllm_o3_mini = json.load(file)
all_module_raw_rtllm_o3mini_df = pd.DataFrame()

In [ ]:
for moduleNormPath in moduleNormPaths:
    moduleName = os.path.basename(moduleNormPath)
    if moduleName in raw_rtllm_o3_mini:
        row_data = np.array(list(raw_rtllm_o3_mini[moduleName].values())) / 5
        cur_module_df = pd.DataFrame([row_data], 
                                 index=pd.Index([moduleName], name='Designs'), 
                                 columns=pd.MultiIndex.from_product([['GPT‑o3 mini + RTLLM '],['Syntax', 'Func.']]))
        all_module_raw_rtllm_o3mini_df = pd.concat([all_module_raw_rtllm_o3mini_df, cur_module_df])
    else:
        print(f'Module "{moduleName}" not found')
all_module_raw_rtllm_o3mini_df

## RTLLM Styler

In [ ]:
all_module_df = pd.DataFrame()

In [ ]:
for moduleNormPath in moduleNormPaths:
    moduleName = os.path.basename(moduleNormPath)
    reportFileDir = os.path.join(moduleNormPath, "reports", f'report_{llm_model}.json')
    if not os.path.isfile(reportFileDir):
        print(f'No report for module "{moduleName}"')
        continue                             

    with open(reportFileDir, 'r') as file:
        reportDict = json.load(file)

    exception_trial = sum(reportDict["exception_trial"]) / 5
    tb_failed_trial = sum(reportDict["tb_failed_trial"]) / 5

    llm_model_alias = llm_model_alias_dict[llm_model] if llm_model in llm_model_alias_dict else llm_model
    cur_module_df = pd.DataFrame([[exception_trial, tb_failed_trial]], 
                                 index=pd.Index([moduleName], name='Designs'), 
                                 columns=pd.MultiIndex.from_product([[llm_model_alias],['Syntax', 'Func.']]))
    all_module_df = pd.concat([all_module_df, cur_module_df])
all_module_df

## RTLLM Composition

In [ ]:
all_module_join_df = all_module_df.join([all_module_raw_rtllm_4omini_df, all_module_raw_rtllm_o3mini_df], how='outer')

In [ ]:
# all_module_join_df = (all_module_join_df * 100).astype(np.int64)

In [ ]:
all_module_join_df

## RTLLM Style

In [ ]:
all_module_join_style_df = all_module_join_df.agg(["mean"]).style \
                   .format(precision=4) \
                   .relabel_index(["Average"])

In [ ]:
all_module_join_style_df.data = all_module_join_style_df.data.round(2)
# all_module_join_style_df.apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen'], axis=1)

In [ ]:
# all_module_join_base_style_df = all_module_join_df.style.apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen'], axis=1)

In [ ]:
all_module_join_style_df = all_module_join_df.style.concat(all_module_join_style_df)

In [ ]:
# all_module_join_style_df.format_index(str.upper, axis=1)
all_module_join_style_df

In [ ]:
# https://saturncloud.io/blog/how-to-format-certain-floating-dataframe-columns-into-percentage-in-pandas/
all_module_join_style_df.data = all_module_join_style_df.data.map('{:.0%}'.format)
all_module_join_style_df.concatenated[0].data = all_module_join_style_df.concatenated[0].data.map('{:.2%}'.format)
all_module_join_style_df

In [ ]:
# https://stackoverflow.com/questions/30499379/python-pandas-to-latex-issues-with-the-backslash
# https://pandas.pydata.org/docs/reference/api/pandas.io.formats.style.Styler.format_index.html#pandas.io.formats.style.Styler.format_index
_ = all_module_join_style_df.format_index(escape="latex")
all_module_join_style_df.concatenated[0].format(escape="latex")
all_module_join_style_df.format(escape="latex")

In [ ]:
def highlight_max(s: pd.core.series.Series, props:list = ['', '']):
    try:
        syntax_list:list = np.array([int(cstring.replace("%", "")) for cstring in s.values[0::2]])
        tb_list:list =np.array([int(cstring.replace("%", "")) for cstring in s.values[1::2]])
    except:
        syntax_list:list = np.array([float(cstring.replace("%", "")) for cstring in s.values[0::2]])
        tb_list:list =np.array([float(cstring.replace("%", "")) for cstring in s.values[1::2]])
    all_hightlight_list = np.array([''] * len(s.values), dtype=object)
    all_hightlight_list[0::2] = np.where(syntax_list == np.nanmax(syntax_list), props[0], '')
    all_hightlight_list[1::2] = np.where(tb_list == np.nanmax(tb_list), props[1], '')
    
    return all_hightlight_list

In [ ]:
all_module_join_style_df.concatenated[0].apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen'], axis=1)
all_module_join_style_df.apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen', float], axis=1)

In [ ]:
all_module_join_style_df.set_table_styles([{'selector': 'toprule', 'props': ':hline;'},
    {'selector': 'midrule', 'props': ':hline;'},
    {'selector': 'bottomrule', 'props': ':hline;'}])
# all_module_join_style_df.set_table_styles([{'selector': 'th.col_heading', 'props': 'text-align: center; font-weight: bold;'}])

In [ ]:
all_module_join_style_df.map_index(
    lambda v: "font-weight: bold;", axis="columns", level=0
)

In [ ]:
latex = all_module_join_style_df.to_latex(convert_css=True, hrules=True, column_format="c|cc|cc|cc")
latex_list = latex.splitlines()
mid_bot_rule_index = (latex_list.index("\\midrule"), latex_list.index("\\bottomrule") - 1)
for i in range(mid_bot_rule_index[0] + 1, mid_bot_rule_index[1]):
    latex_list[i] += " \\midrule"

with open(f'reports/tables/{llm_model}.tex', 'w+') as file:
    print('\n'.join(latex_list), file=file)

## COMBA-PROMPT Fix rate

### RTLLM

In [ ]:
descriptionType = "RTLLM.txt"
with open(f'reports/fixrate/fixrate.{descriptionType}.json', 'r') as file:
    raw_fixrate_RTLLM_4o_mini = json.load(file)
all_module_raw_raw_fixrate_RTLLM_4omini_df = pd.DataFrame()

In [ ]:
for moduleNormPath in moduleNormPaths:
    moduleName = os.path.basename(moduleNormPath)
    if moduleName in raw_fixrate_RTLLM_4o_mini:
        row_data = np.array(list(raw_fixrate_RTLLM_4o_mini[moduleName].values()))
        # row_data = np.array(list(raw_rtllm_4o_mini[moduleName].values())) / 5
        cur_module_df = pd.DataFrame([row_data], 
                                 index=pd.Index([moduleName], name='Designs'), 
                                 columns=pd.MultiIndex.from_product([['GPT‑4o mini + RTLLM '],['Syntax', 'Func.']]))
        all_module_raw_raw_fixrate_RTLLM_4omini_df = pd.concat([all_module_raw_raw_fixrate_RTLLM_4omini_df, cur_module_df])
    else:
        print(f'Module "{moduleName}" not found')
all_module_raw_raw_fixrate_RTLLM_4omini_df

### COMPA-PROMPT

In [ ]:
descriptionType = "xml"
with open(f'reports/fixrate/fixrate.{descriptionType}.json', 'r') as file:
    raw_fixrate_comba_prompt_4o_mini = json.load(file)
all_module_raw_raw_fixrate_comba_prompt_4omini_df = pd.DataFrame()

In [ ]:
for moduleNormPath in moduleNormPaths:
    moduleName = os.path.basename(moduleNormPath)
    if moduleName in raw_fixrate_comba_prompt_4o_mini:
        row_data = np.array(list(raw_fixrate_comba_prompt_4o_mini[moduleName].values()))
        # row_data = np.array(list(raw_rtllm_4o_mini[moduleName].values())) / 5
        cur_module_df = pd.DataFrame([row_data], 
                                 index=pd.Index([moduleName], name='Designs'), 
                                 columns=pd.MultiIndex.from_product([['GPT‑4o mini + COMBA-PROMPT '],['Syntax', 'Func.']]))
        all_module_raw_raw_fixrate_comba_prompt_4omini_df = pd.concat([all_module_raw_raw_fixrate_comba_prompt_4omini_df, cur_module_df])
    else:
        print(f'Module "{moduleName}" not found')
all_module_raw_raw_fixrate_comba_prompt_4omini_df

### Composition

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_df = all_module_raw_raw_fixrate_comba_prompt_4omini_df.join([all_module_raw_raw_fixrate_RTLLM_4omini_df], how='outer')

In [ ]:
# all_module_join_df = (all_module_join_df * 100).astype(np.int64)

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_df

### Style

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df = all_module_raw_raw_fixrate_comba_prompt_4omini_df.agg(["mean"]).style \
                   .format(precision=4) \
                   .relabel_index(["Average"])

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.data = all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.data.astype(np.float64)
# all_module_join_style_df.apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen'], axis=1)

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df = all_module_raw_raw_fixrate_comba_prompt_4omini_df.style.concat(all_module_raw_raw_fixrate_comba_prompt_4omini_style_df)

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.format_index(str.upper, axis=1)
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df

In [ ]:
# https://saturncloud.io/blog/how-to-format-certain-floating-dataframe-columns-into-percentage-in-pandas/
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.data = all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.data.map('{:.2%}'.format)
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.concatenated[0].data = all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.concatenated[0].data.map('{:.2%}'.format)
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df

In [ ]:
# https://stackoverflow.com/questions/30499379/python-pandas-to-latex-issues-with-the-backslash
# https://pandas.pydata.org/docs/reference/api/pandas.io.formats.style.Styler.format_index.html#pandas.io.formats.style.Styler.format_index
_ = all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.format_index(escape="latex")
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.concatenated[0].format(escape="latex")
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.format(escape="latex")

In [ ]:
def highlight_max(s: pd.core.series.Series, props:list = ['', '']):
    try:
        syntax_list:list = np.array([int(cstring.replace("%", "")) for cstring in s.values[0::2]])
        tb_list:list =np.array([int(cstring.replace("%", "")) for cstring in s.values[1::2]])
    except:
        syntax_list:list = np.array([float(cstring.replace("%", "")) for cstring in s.values[0::2]])
        tb_list:list =np.array([float(cstring.replace("%", "")) for cstring in s.values[1::2]])
    all_hightlight_list = np.array([''] * len(s.values), dtype=object)
    all_hightlight_list[0::2] = np.where(syntax_list == np.nanmax(syntax_list), props[0], '')
    all_hightlight_list[1::2] = np.where(tb_list == np.nanmax(tb_list), props[1], '')
    
    return all_hightlight_list

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.concatenated[0].apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen'], axis=1)
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.apply(highlight_max, props=['color:black;background-color:lightblue', 'color:black;background-color:lightgreen', float], axis=1)

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.set_table_styles([{'selector': 'toprule', 'props': ':hline;'},
    {'selector': 'midrule', 'props': ':hline;'},
    {'selector': 'bottomrule', 'props': ':hline;'}])
# all_module_join_style_df.set_table_styles([{'selector': 'th.col_heading', 'props': 'text-align: center; font-weight: bold;'}])

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.map_index(
    lambda v: "font-weight: bold;", axis="columns", level=0
)

In [ ]:
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.data = all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.data.map(lambda x: x.replace('.00', ''))
all_module_raw_raw_fixrate_comba_prompt_4omini_style_df

In [ ]:
latex = all_module_raw_raw_fixrate_comba_prompt_4omini_style_df.to_latex(convert_css=True, hrules=True, column_format="c|cc|cc") # "c|cc|cc|cc"
latex_list = latex.splitlines()
mid_bot_rule_index = (latex_list.index("\\midrule"), latex_list.index("\\bottomrule") - 1)
for i in range(mid_bot_rule_index[0] + 1, mid_bot_rule_index[1]):
    latex_list[i] += " \\midrule"

with open(f'reports/tables/{llm_model}.{descriptionType}.tex', 'w+') as file:
    print('\n'.join(latex_list), file=file)

# Glob - File list

In [ ]:
import os, glob

In [ ]:
glob.glob('modules/adder_8bit')

# XML Description

In [ ]:
!pip install pydantic-xml[lxml]

In [ ]:
import pathlib

In [ ]:
%%writefile scripts/xmlDescription.py
# Genterated automatically by /run.ipynb
from pydantic_xml import BaseXmlModel, attr, element, RootXmlModel
from pydantic import Field
from typing import Optional, Union, List, Literal

class IDModel(BaseXmlModel):
    id: str = attr()
class IO(IDModel):
    description: Optional[str] = None
    width_description: Optional[str] = attr(default=None)
class Ports(BaseXmlModel, tag="ports"):
    input: List[IO] = element()
    output: List[IO] = element()

class Parameter(IDModel):
    description: Optional[str] = None
class ParameterDescription(BaseXmlModel, tag="parameter_description"):
    parameter: List[Parameter] = element()

class PartialLogicDescription(IDModel):
    description: Optional[str] = None
    width_description: Optional[str] = attr(default=None)
    depth_description: Optional[str] = attr(default=None)
    type: Literal["combinational_logic", "combinational_logic_operation", "sequential_logic", "sequential_logic_operation"] = attr()
    
class LogicDescription(BaseXmlModel, tag="logic_description", search_mode='unordered'):
    description: str = element()
    logic: List[PartialLogicDescription] = element(default=None)
    
class Module(IDModel, tag='module'):
    description: str = element(default=None)
    ports: Optional[Ports] = element(default=None)
    parameter_description: Optional[ParameterDescription] = element(default=None)
    logic_description: Optional[LogicDescription] = element(default=None)
    implementation: str = element()
    task: str = element(default="Give me the complete Verilog code.")
class Modules(RootXmlModel, tag='modules'):
    root: List[Module]

In [ ]:
%run scripts/xmlDescription.py

In [ ]:
Modules.va

In [ ]:
xml_doc = pathlib.Path('./test/test.xml').read_text()
xml_doc
xml_module = Modules.from_xml(xml_doc)
xml_module

In [ ]:
json_doc = pathlib.Path('./test/export.xml')
json_doc.write_text(xml_module.to_xml(pretty_print=True).decode('utf-8'))
# xml_module.to_xml(encoding='utf-8').decode('utf-8')
# json_doc.write_text(xml_module.to_xml().decode('utf-8'))

In [ ]:
json_doc = pathlib.Path('./test/test.json')
json_doc.write_text(xml_module.model_dump_json(indent=4))

In [ ]:
# from_json?


# Hashlib

# Code Agent

In [ ]:
%%writefile scripts/codeAgent.py
class LLMCodeAgent:
    def __init__(self, modulePath:str):
        pass
    def __iter__(self):
        self.a = 1
        self.status = "error"
        return self
    def __next__(self):
        confirm = input(f"Next? (y/n) ")
        if confirm == "n" or confirm == "N":
            print("End Agent")
            raise StopIteration

        x = self.a
        self.a += 1
        return (x, self.status)

    def __call__(self):
        print("Start agent")
        myiter = iter(self)
        currentStatus = None
        for x in myiter:
            currentStatus = x
            print("iter status", currentStatus[1], ". iter times: ", currentStatus[0])

        print("last status", currentStatus[1], ". iter times: ", currentStatus[0])


In [ ]:
import os
os._exit(00)

In [ ]:
%run scripts/codeAgent.py

In [ ]:
from scripts.codeAgent import LLMCodeAgent

llmCodeAgent = LLMCodeAgent('modules/adder_8bit')

In [ ]:
llmCodeAgent.pnggraph()

In [ ]:
import json
jsontxt = """
Warns that an instance has a pin that is connected to `.pin_name()`, e.g., not another signal, but with an explicit mention of the pin. It may be desirable to disable PINCONNECTEMPTY, as this indicates the intention to have a no-connect.
"""
a = {'txt': jsontxt}
json.dump(a, open('.ignore/tmp.txt', 'w+'))

In [ ]:
import json
jsontxt = """A Warning that the specified signal is never used/consumed.
For example, you have a wire called `a`, declared as follows:

```sv
wire [3:0] a;
```

But in the operation of a module, you only use 2/4 bits of the wire `a`. For example:

```sv
assign a[1:0] = {1'b0, 1'b1};
```

And you forgot to use the `a[3:2]`. So, that is why this warning is raised.

You should fix this warning by removing unused signals in the declaration of a port, such as:

```sv
wire [1:0] a;
```

Or you can make use of the unused signals that you forgot:

```sv
assign a[3:0] = {1'b0, 1'b1, 1'b0, 1'b1};
```

"""
a = {'txt': jsontxt}
json.dump(a, open('.ignore/tmp.txt', 'w+'))

In [ ]:
import ast
from scripts.lazy import readFileContent

verilator_warns = readFileContent("rag/verilator_warns.json")
verilator_warns: dict = ast.literal_eval(verilator_warns)

print(verilator_warns["WIDTHEXPAND"])

# VCD Parser

In [ ]:
!pip install vcdvcd

In [ ]:
from vcdvcd import VCDVCD

In [ ]:

# Do the parsing.
vcd = VCDVCD('waveform.vcd')

# List all human readable signal names.
print(vcd.references_to_ids.keys())

In [ ]:
# View all signal data.
print(vcd.data)